<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/6_catboost_model/6_1_baseline_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **6_1_baseline_model**

## Introducción y Resumen

CatBoost (Categorical Boosting) es un algoritmo de gradient boosting sobre árboles de decisión, diseñado para ofrecer alto rendimiento predictivo con mínima necesidad de preprocesamiento. Se caracteriza por una implementación robusta frente al overfitting, estabilidad en datasets ruidosos y una buena capacidad para capturar relaciones no lineales entre variables.

A diferencia de otros métodos de boosting, CatBoost incorpora técnicas internas que reducen el sesgo en el entrenamiento y mejoran la generalización, lo que lo hace especialmente adecuado para problemas de series temporales transformadas en features tabulares, como nuestro caso con datos intradía.

**Aplicación al horizonte de 90 minutos**

En el contexto de nuestro proyecto, CatBoost se utilizará para predecir el retorno acumulado a 90 minutos, a partir de:

- Ventanas deslizantes de datos intradía (OHLCV).
- Factores técnicos y alpha factors ya ingenierizados.
- Observaciones independientes por día de trading (sin continuidad temporal entre jornadas).

Bajo este enfoque, CatBoost actúa como un modelo supervisado de regresión, donde:
- Entrada (X): features agregadas o vectorizadas de la ventana histórica.
- Salida (y): target_return_90.

El modelo no asume estructura temporal explícita, sino que aprende patrones estadísticos que anticipan el comportamiento del precio a 90 minutos, lo cual es consistente con tu formulación actual del problema.

**Ventajas clave para este proyecto**

- Excelente desempeño en datasets tabulares con muchas features.
- Baja sensibilidad al escalado de variables.
- Buena robustez frente a ruido intradía.
- Entrenamiento eficiente y estable.
- Ideal como baseline fuerte frente a modelos más complejos (LSTM, Transformer, TFT).




## 0. Configuración del Entorno


### 0.1. Clonado de repositorio / Acceso a Drive

In [1]:
#Clonamos el repo
#LINK DE REPOSITORIO: https://github.com/GUNAPILLCO/neural_profit
#!git clone https://github.com/GUNAPILLCO/neural_profit.git

In [2]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### 0.2. Instalación de librerías


In [3]:
%pip uninstall -y catboost -q

# Alineamos el stack numérico y CatBoost en una sola transacción
%pip install --no-cache-dir -U \
  numpy==2.1.2 \
  scipy==1.13.1 \
  scikit-learn==1.5.2 \
  catboost==1.2.8 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 263.5 MB/s eta 0:00:00


In [4]:
import catboost
from catboost import CatBoostRegressor, Pool

### 0.3. Importación de librerías


In [5]:
# ==============================
# Librerías estándar de Python
# ==============================
import os
import sys
import re
import glob
import warnings
import requests
from datetime import datetime, timedelta
from functools import reduce

# ==============================
# Manejo y procesamiento de datos
# ==============================
import pandas as pd
import numpy as np
from tabulate import tabulate

# ==============================
# Visualización
# ==============================
import matplotlib.pyplot as plt

# ==============================
# Estadística
# ==============================
from scipy.stats import spearmanr

# ==============================
# Machine Learning y utilidades
# ==============================
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import sklearn, scipy
import sklearn, numpy, scipy #optuna
#from sklearn.ensemble import RandomForestRegressor

import joblib
#import optuna
from tqdm import tqdm

# ==============================
# Configuración general
# ==============================
warnings.filterwarnings("ignore")

import time

import xgboost as xgb
#from xgboost import XGBRegressor

import sys, platform, lightgbm as lgb
import numpy as np, pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
#from lightgbm import LGBMRegressor


In [ ]:
print("python:", sys.version)
print("Platform:", platform.platform())
print("numpy:", numpy.__version__)
print("scipy:", scipy.__version__)
print("sklearn:", sklearn.__version__)
#print("optuna:", optuna.__version__)
print("xgboost:", xgb.__version__)
print("lightgbm:", lgb.__version__)

# CatBoost usa la clase para exponer versión
print("catboost:", catboost.__version__)

## 1. Carga de datos

### 1.1. Carga de datasets `mnq_train`, `mnq_valid` y `mnq_test`






In [6]:
def load_data(data: str):

    data_path = f'{drive_path}/3_dataset_preparation/mnq_{data}.parquet'
    # Leer el archivo Parquet y cargarlo en un DataFrame
    df = pd.read_parquet(data_path)

    # Asegurar que el índice esté en formato datetime
    df.index = pd.to_datetime(df.index)

    # Crear una nueva columna 'date' con la fecha extraída del índice
    df['date'] = df.index.date

    # Reordenar columnas: 'date', 'time_str', y luego el resto
    cols = ['date'] + [col for col in df.columns if col not in ['date']]

    df = df[cols]

    return df

In [7]:
#mnq_train = load_data("train")
#mnq_valid = load_data("valid")
#mnq_test = load_data("test")

### 1.2. Información de datasets


In [8]:
def info_dataset(df, name: str):
  print(f"Información del dataset {name}:\n")

  # Contar valores únicos en la columna 'date'
  num_dias = df['date'].nunique()
  print(f"\tCantidad de días: {num_dias}")

  # Filtrar valores válidos
  validos_por_dia = df.dropna(subset=['close']).groupby('date').size()

  # Calcular el promedio
  promedio_por_fecha = validos_por_dia.mean()
  print(f"\tRegistros por día: {int(promedio_por_fecha)}")

  primer_hora = df.index[0].strftime('%H:%M')
  ultima_hora = df.index[-1].strftime('%H:%M')
  zona_horaria = df.index[0].tzinfo


  print(f"\tHora diaria de inicio {primer_hora}")
  print(f"\tHora diaria de final {ultima_hora}")
  print(f"\tZona horaria: {zona_horaria}\n")

  return num_dias, promedio_por_fecha

In [9]:
#info_dataset(mnq_train, 'mnq_train')
#info_dataset(mnq_valid, 'mnq_valid')
#info_dataset(mnq_test, 'mnq_test')

### 1.3. Carga de listado de features por ventana de tiempo

In [10]:
import json

# Ruta al archivo guardado
path = f'{drive_path}/2_feature_engineering/features_list.json'

with open(path, "r") as f:
    features_dict = json.load(f)

# Extraer las listas
#features_to_30 = features_dict["features_to_30"]
#features_to_60 = features_dict["features_to_60"]
#features_to_90 = features_dict["features_to_90"]


In [11]:
#print(f'Listado de features para 30min: {features_to_30}')
#print(f'Listado de features para 60min: {features_to_60}')
#print(f'Listado de features para 90min: {features_to_90}')

## **2. Carga de ventanas `X_train_*_scaled`, `X_valid_*_scaled`, `X_test_*_scaled`**

In [12]:
#Lista de K folds
k_folds = [1, 2, 3, 4, 5]

### 2.0. Funciones

#### Función para cargar ventanas

In [13]:
def load_windows_and_scaler(k: str, scaled=True):
    """
    Carga datasets (X, y) para train, valid y test junto con el scaler global.

    Parámetros
    ----------
    drive_path : str
        Ruta base donde se encuentran los archivos.
    scaled : bool, default=True
        Si True busca en la carpeta 'ventanas_x_y_scaled',
        si False en 'ventanas_x_y'.

    Retorna
    -------
    X_train, y_train, X_valid, y_valid, X_test, y_test, scaler
    """

    #Ruta de ventandas escaladas que usamos para transformer
    path_train  = f'{drive_path}/5_transformer_model/5_2_k_scaler/fold_{k}/X_train_sc_{k}.npz'
    path_valid  = f'{drive_path}/5_transformer_model/5_2_k_scaler/fold_{k}/X_valid_sc_{k}.npz'
    path_test   = f'{drive_path}/5_transformer_model/5_2_k_scaler/fold_{k}/X_test_sc_{k}.npz'

    #Ruta de escalador
    path_scaler = f"{drive_path}/5_transformer_model/5_2_k_scaler/global_scaler.pkl"

    # Cargar npz
    data_train = np.load(path_train)
    data_valid = np.load(path_valid)
    data_test  = np.load(path_test)

    # Extraer X, y
    X_train, y_train = data_train["X"], data_train["y"]
    print(f'\tX_train_sc_{k} e y_train_{k} extraídos correctamente')
    X_valid, y_valid = data_valid["X"], data_valid["y"]
    print(f'\tX_valid_sc_{k} e y_valid_{k} extraídos correctamente')
    X_test,  y_test  = data_test["X"],  data_test["y"]
    print(f'\tX_test_sc_{k} e y_test_{k} extraídos correctamente')

    # Cargar scaler
    scaler = joblib.load(path_scaler)

    return X_train, y_train, X_valid, y_valid, X_test, y_test, scaler

#### Función para revisar información de ventanas

In [14]:
def xy_info(k, X_train, y_train, X_valid, y_valid, X_test, y_test, silent=False):
    import numpy as np
    import psutil

    if not silent:
        print(f"Información de {k}:")
        print("----------------------------------------")

    # Memoria total
    total_ram_gb = psutil.virtual_memory().total / (1024 ** 3)

    def print_set_info(nombre, X, y):
        if silent:
            return  # No imprimir nada

        n_samples = X.shape[0]
        size_X_gb = X.nbytes / (1024 ** 3)
        size_y_gb = y.nbytes / (1024 ** 3)
        total_gb = size_X_gb + size_y_gb
        perc_ram = (total_gb / total_ram_gb) * 100
        y_flat = np.ravel(y)

        print(f"Set de {nombre}:")
        print(f"\t{n_samples} ventanas")
        print(f"\tTamaño X: {size_X_gb:.3f} GB")
        print(f"\tTamaño y: {size_y_gb:.6f} GB")
        print(f"\tTOTAL: {total_gb:.3f} GB → {perc_ram:.1f}% RAM\n")

    # Mostrar info solo si silent=False
    print_set_info("entrenamiento", X_train, y_train)
    print_set_info("validación",    X_valid, y_valid)
    print_set_info("testeo",        X_test,  y_test)

    # Pesos = cantidad de ventanas
    w_train = X_train.shape[0]
    w_valid = X_valid.shape[0]
    w_test  = X_test.shape[0]

    return w_train, w_valid, w_test


### 2.1. Carga de ventanas

In [15]:
#X_train_90_scaled, y_train_90, X_valid_90_scaled, y_valid_90, X_test_90_scaled, y_test_90, scaler_90 = load_windows_and_scaler(target = '90')

# Diccionarios para almacenar datos escalados por fold
X_train_sc = {}
y_train_sc = {}
X_valid_sc = {}
y_valid_sc = {}
X_test_sc  = {}
y_test_sc  = {}
scalers    = {}

for k in k_folds:
    print(f'Fold {k}:')

    X_train, y_train, X_valid, y_valid, X_test, y_test, scaler = load_windows_and_scaler(k)

    # Guardar todo en diccionarios
    X_train_sc[k] = X_train
    y_train_sc[k] = y_train

    X_valid_sc[k] = X_valid
    y_valid_sc[k] = y_valid

    X_test_sc[k]  = X_test
    y_test_sc[k]  = y_test

    scalers[k] = scaler

    print(f"  - Datos escalados cargados y almacenados en diccionarios.")
    print("-" * 40)

Fold 1:
	X_train_sc_1 e y_train_1 extraídos correctamente
	X_valid_sc_1 e y_valid_1 extraídos correctamente
	X_test_sc_1 e y_test_1 extraídos correctamente
  - Datos escalados cargados y almacenados en diccionarios.
----------------------------------------
Fold 2:
	X_train_sc_2 e y_train_2 extraídos correctamente
	X_valid_sc_2 e y_valid_2 extraídos correctamente
	X_test_sc_2 e y_test_2 extraídos correctamente
  - Datos escalados cargados y almacenados en diccionarios.
----------------------------------------
Fold 3:
	X_train_sc_3 e y_train_3 extraídos correctamente
	X_valid_sc_3 e y_valid_3 extraídos correctamente
	X_test_sc_3 e y_test_3 extraídos correctamente
  - Datos escalados cargados y almacenados en diccionarios.
----------------------------------------
Fold 4:
	X_train_sc_4 e y_train_4 extraídos correctamente
	X_valid_sc_4 e y_valid_4 extraídos correctamente
	X_test_sc_4 e y_test_4 extraídos correctamente
  - Datos escalados cargados y almacenados en diccionarios.
-------------

In [16]:
pesos_folds = {}

for k in k_folds:
    w_train, w_valid, w_test = xy_info(
        k,
        X_train_sc[k],
        y_train_sc[k],
        X_valid_sc[k],
        y_valid_sc[k],
        X_test_sc[k],
        y_test_sc[k],
        silent=True   # evita imprimir
    )

    pesos_folds[k] = {
        "w_train": w_train,
        "w_valid": w_valid,
        "w_test":  w_test,
    }


## **3. Dataset de Métricas**

Dado que cada entrenamiento demanda un tiempo considerable, antes de proceder verificaremos si ya existe un resultado previo de este modelo consultando el dataset de métricas.

### 3.1. Función para cargar métricas o generar dataset

In [17]:
def load_metrics(subcarpeta: str, data: str):
    data_path = f'{drive_path}/6_catboost_model/{subcarpeta}/{data}.parquet'
    # Leer el archivo Parquet y cargarlo en un DataFrame
    df = pd.read_parquet(data_path)
    return df

In [18]:
def metrics_verify(subcarpeta: str, data: str) -> bool:
    data_path = f'{drive_path}/v_model/{subcarpeta}/{data}.parquet'
    return os.path.exists(data_path)


In [19]:
def load_or_create_metrics (subcarpeta: str, data:str):
  if metrics_verify(subcarpeta, data):
      print(f"Las métricas existen y son almacenadas en {data[2:len(data)]}")
      model_metrics = load_metrics(subcarpeta, data)
      metrics = True
  else:
      print(f"Las métricas no existen. Se crea el dataset {data[2:len(data)]} para almacenar las métricas")
      #Creamos la tabla para almacenar las métricas
      model_metrics = pd.DataFrame(columns=["RMSE", "MAE", "R2", "SMAPE", "DirAcc"])
      metrics = False

  return model_metrics, metrics

In [20]:
baseline_folds_metrics, flag_baseline_folds_metrics = load_or_create_metrics("6_1_baseline_model", "0_baseline_folds_metrics")
baseline_metrics, flag_baseline_metrics = load_or_create_metrics("6_1_baseline_model", "1_baseline_metrics")

Las métricas no existen. Se crea el dataset baseline_folds_metrics para almacenar las métricas
Las métricas no existen. Se crea el dataset baseline_metrics para almacenar las métricas


### 3.2. Función para guardar métricas

In [21]:
def save_metrics (metrics,  subcarpeta: str, metrics_name: str):
  metrics_path = f"{drive_path}/6_catboost_model/{subcarpeta}/{metrics_name}.parquet"
  metrics.to_parquet(metrics_path, index = True)
  print(f"Métricas guardadas en {metrics_path}")

### 3.3. Función para calcular las métricas

In [22]:
def evaluate_model(model, X, y_true, y_pred=None, eps=1e-8):
    """
    Evalúa RMSE, MAE, R2, SMAPE y DirAcc.
    - Si y_pred es None, predice con el modelo usando X.
    - Evita mean_squared_error(squared=...) para máxima compatibilidad.
    """
    if y_pred is None:
        y_pred = model.predict(X)

    # Asegurar 1D
    y_true = np.ravel(y_true)
    y_pred = np.ravel(y_pred)

    # RMSE sin sklearn
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    mae = float(mean_absolute_error(y_true, y_pred))
    r2  = float(r2_score(y_true, y_pred))

    # SMAPE
    smape_val = 100.0 * np.mean(
        (np.abs(y_true - y_pred) / ((np.abs(y_true) + np.abs(y_pred)) / 2.0 + eps))
    )

    # Directional Accuracy
    directional_acc = float(np.mean(np.sign(y_true) == np.sign(y_pred)))

    return {
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        "SMAPE": float(smape_val),
        "DirAcc": directional_acc
    }

In [23]:
def print_metrics(metrics, target:str):
  print(f"Métricas de {target}:\n")
  for k, v in metrics.items():
      print(f"\t{k:>5}:\t {float(v):.6f}")

## **4. Definición de modelo**


### **4.1. Función de entrenamiento para modelo**

In [24]:
from catboost import CatBoostRegressor, Pool

def train_model_catboost(
    best_params,
    X_train, y_train,
    X_valid, y_valid,
    *,
    cat_features=None,
    use_gpu=False,
    iterations=5000,
    od_wait=200,
    verbose_every=200
):
    """
    Entrena un modelo CatBoostRegressor usando early stopping.

    NOTA IMPORTANTE:
    - NO se usan pesos de entrenamiento.
    - Los pesos por fold se utilizarán únicamente para ponderar métricas
      en la etapa de evaluación final.
    """

    # ------------------------------------------------------------------
    # 1. Copia de hiperparámetros
    # ------------------------------------------------------------------
    # Se copia el diccionario para no modificar el original
    params = dict(best_params or {})

    # ------------------------------------------------------------------
    # 2. Defaults de CatBoost (si no vienen definidos)
    # ------------------------------------------------------------------
    params.setdefault("loss_function", "RMSE")   # función de pérdida
    params.setdefault("eval_metric", "RMSE")     # métrica de validación
    params.setdefault("random_seed", 42)         # reproducibilidad
    params.setdefault("use_best_model", True)    # conserva la mejor iteración
    params.setdefault("od_type", "Iter")         # early stopping por iteraciones
    params.setdefault("od_wait", od_wait)        # paciencia del early stopping
    params.setdefault("verbose", verbose_every)  # frecuencia de logs
    params.setdefault("thread_count", -1)        # usa todos los cores
    params.setdefault("iterations", iterations) # máximo de árboles
    params.setdefault("allow_writing_files", False)  # evita archivos temporales

    # ------------------------------------------------------------------
    # 3. Configuración opcional de GPU
    # ------------------------------------------------------------------
    if use_gpu:
        params["task_type"] = "GPU"
        # params["devices"] = "0"  # opcional, si querés fijar GPU

    # ------------------------------------------------------------------
    # 4. Creación de Pools
    # ------------------------------------------------------------------
    # Pool es la estructura interna eficiente de CatBoost
    # Aquí NO se pasan pesos
    train_pool = Pool(
        X_train,
        y_train,
        cat_features=cat_features
    )

    valid_pool = Pool(
        X_valid,
        y_valid,
        cat_features=cat_features
    )

    # ------------------------------------------------------------------
    # 5. Entrenamiento del modelo
    # ------------------------------------------------------------------
    model = CatBoostRegressor(**params)
    model.fit(
        train_pool,
        eval_set=valid_pool
    )

    # ------------------------------------------------------------------
    # 6. Predicción sobre validación
    # ------------------------------------------------------------------
    # Con use_best_model=True, predict() usa automáticamente
    # la mejor iteración encontrada en validación
    preds_valid = model.predict(valid_pool)

    # ------------------------------------------------------------------
    # 7. Salida
    # ------------------------------------------------------------------
    return model, preds_valid


### **4.2. Hiperparámetros baseline para modelo**


In [25]:
# ------------------------------------------------------------------
# Parámetros baseline para CatBoost (regresión)
# Horizonte objetivo: retorno a 90 minutos
# Configuración pensada para entrenamiento en GPU
# ------------------------------------------------------------------

catboost_baseline_params = {
    # --------------------------------------------------------------
    # Función de pérdida y métrica de evaluación
    # --------------------------------------------------------------
    "loss_function": "RMSE",   # Optimiza el error cuadrático medio
    "eval_metric": "RMSE",     # Métrica usada para early stopping

    # --------------------------------------------------------------
    # Parámetros de boosting
    # --------------------------------------------------------------
    "learning_rate": 0.02,     # Tasa de aprendizaje baja para mayor estabilidad
    "depth": 8,                # Profundidad de los árboles (capacidad del modelo)
    "l2_leaf_reg": 3.0,        # Regularización L2 para reducir overfitting

    # --------------------------------------------------------------
    # Estocasticidad (compatible con GPU)
    # --------------------------------------------------------------
    # "rsm": 0.8,               # Submuestreo de features (NO compatible con GPU)
    # "subsample": 0.8,         # Bootstrap clásico (NO recomendado en GPU)
    "bagging_temperature": 1.0,  # Bayesian bootstrap (sí compatible con GPU)
    "random_strength": 1.0,       # Ruido en la selección de splits

    # --------------------------------------------------------------
    # Reproducibilidad
    # --------------------------------------------------------------
    "random_seed": 42,         # Semilla fija para resultados reproducibles

    # --------------------------------------------------------------
    # Early stopping
    # --------------------------------------------------------------
    "use_best_model": True,    # Conserva el mejor modelo según validación
    "od_type": "Iter",         # Early stopping basado en iteraciones
    "od_wait": 200,            # Paciencia del early stopping

    # --------------------------------------------------------------
    # Configuración de hardware
    # --------------------------------------------------------------
    "task_type": "GPU",        # Entrenamiento en GPU

    # --------------------------------------------------------------
    # Logging
    # --------------------------------------------------------------
    "verbose": 200,            # Frecuencia de impresión del progreso
}


### **4.3. Función conjunta**

In [26]:
def run_catboost_experiment(
    metrics_flag: bool,
    model_key: str,
    X_train_scaled,
    y_train,
    X_valid_scaled,
    y_valid,
    catboost_params: dict,
    catboost_metrics_df: pd.DataFrame | None = None,
    *,
    # extras específicos de CatBoost (opcionales)
    cat_features=None,          # índices o nombres de columnas categóricas
    use_gpu: bool = False,
    iterations: int = 5000,
    od_wait: int = 200,
    verbose_every: int = 200,
):
    """
    Ejecuta un experimento CatBoost SIN subsampleo (usa todo train/valid).

    Parámetros
    ----------
    metrics_flag : bool
        Si True, no entrena y busca métricas previas en catboost_metrics_df[model_key].
    model_key : str
        Nombre/índice del modelo en la tabla de métricas (ej: 'CAT_90_fold_1').
    X_train_scaled, y_train : arrays
        Dataset de entrenamiento (ya escalado si corresponde).
    X_valid_scaled, y_valid : arrays
        Dataset de validación (ya escalado si corresponde).
    catboost_params : dict
        Hiperparámetros nativos de CatBoost (depth, learning_rate, l2_leaf_reg, etc.).
    catboost_metrics_df : pd.DataFrame | None
        DataFrame de métricas para leer/escribir (index por model_key). Opcional.

    Retorna
    -------
    dict
        Diccionario con métricas {"RMSE","MAE","R2","SMAPE","DirAcc"}.
    """

    # ------------------------------------------------------------
    # 1) Si ya hay métricas guardadas y metrics_flag=True, devolverlas
    # ------------------------------------------------------------
    if (
        metrics_flag is True
        and catboost_metrics_df is not None
        and model_key in catboost_metrics_df.index
    ):
        print("El modelo fue entrenado anteriormente y las métricas ya fueron calculadas\n")
        metrics_dict = catboost_metrics_df.loc[model_key].to_dict()
        print_metrics(metrics_dict, model_key)
        return metrics_dict

    # ------------------------------------------------------------
    # 2) Entrenar usando todo el train/valid (sin subsampleo)
    # ------------------------------------------------------------
    print(f"Entrenando modelo {model_key} (sin resample)...")

    model, y_pred_valid = train_model_catboost(
        best_params=catboost_params,
        X_train=X_train_scaled, y_train=y_train,
        X_valid=X_valid_scaled, y_valid=y_valid,
        cat_features=cat_features,
        use_gpu=use_gpu,
        iterations=iterations,
        od_wait=od_wait,
        verbose_every=verbose_every
    )

    # ------------------------------------------------------------
    # 3) Evaluar en validación
    # ------------------------------------------------------------
    metrics_dict = evaluate_model(model, X_valid_scaled, y_valid, y_pred=y_pred_valid)

    # ------------------------------------------------------------
    # 4) Mostrar métricas
    # ------------------------------------------------------------
    print_metrics(metrics_dict, model_key)

    # ------------------------------------------------------------
    # 5) Guardar métricas si se pasó el DataFrame
    # ------------------------------------------------------------
    if catboost_metrics_df is not None:
        catboost_metrics_df.loc[model_key] = metrics_dict

    return metrics_dict


## **5. Entrenamiento**

In [27]:
if flag_baseline_folds_metrics:
      print("flag_baseline_folds_metrics=True → se omite el entrenamiento de todos los folds.")
else:
      print("flag_baseline_metrics=False → se inicia entrenamiento por folds.")

      for k in k_folds:
          #Definir una clave de modelo por fold (para registro de métricas)
          print(f"Fold {k}:")
          model_key = f"CAT_baseline_fold_{k}"

          # Si ya existe en la tabla de métricas, omitimos SOLO ese fold
          if ("baseline_folds_metrics" in globals()
              and baseline_folds_metrics is not None
              and model_key in baseline_folds_metrics.index):
              print(f"Omitimos este entrenamiento: {model_key} ya existe en baseline_folds_metrics")
              continue

          print(f"\n=== Entrenando modelo: {model_key} ===")
          metrics_k = run_catboost_experiment(
              metrics_flag=flag_baseline_folds_metrics,
              model_key=model_key,
              X_train_scaled=X_train_sc[k],
              y_train=y_train_sc[k],
              X_valid_scaled=X_valid_sc[k],
              y_valid=y_valid_sc[k],
              catboost_params=catboost_baseline_params,
              catboost_metrics_df=baseline_folds_metrics,
              use_gpu=True
              )

flag_baseline_metrics=False → se inicia entrenamiento por folds.
Fold 1:

=== Entrenando modelo: CAT_baseline_fold_1 ===
Entrenando modelo CAT_baseline_fold_1 (sin resample)...
0:	learn: 0.0050748	test: 0.0058573	best: 0.0058573 (0)	total: 96.4ms	remaining: 8m 1s
200:	learn: 0.0032823	test: 0.0043493	best: 0.0043493 (200)	total: 8.39s	remaining: 3m 20s
400:	learn: 0.0029473	test: 0.0042529	best: 0.0042528 (396)	total: 20.2s	remaining: 3m 51s
600:	learn: 0.0027462	test: 0.0042200	best: 0.0042200 (600)	total: 28.3s	remaining: 3m 27s
800:	learn: 0.0025969	test: 0.0042019	best: 0.0042019 (800)	total: 36.3s	remaining: 3m 10s
1000:	learn: 0.0024727	test: 0.0041825	best: 0.0041824 (999)	total: 45.1s	remaining: 3m
1200:	learn: 0.0023672	test: 0.0041703	best: 0.0041699 (1197)	total: 54.4s	remaining: 2m 52s
1400:	learn: 0.0022776	test: 0.0041599	best: 0.0041598 (1398)	total: 1m 3s	remaining: 2m 43s
1600:	learn: 0.0021950	test: 0.0041512	best: 0.0041512 (1599)	total: 1m 14s	remaining: 2m 37s
1800

## **6. Métricas**

In [28]:
if flag_baseline_folds_metrics:
    print("flag_baseline_folds_metrics=True → ya existen métricas del entrenamiento.")
else:
    print("flag_baseline_folds_metrics=False → se inicia entrenamiento por folds.")
    cols_base = ["RMSE", "MAE", "R2", "SMAPE", "DirAcc"]
    # columnas que efectivamente existen en el DataFrame
    cols_to_drop = [c for c in cols_base if c in baseline_folds_metrics.columns]
    # eliminar solo si hay columnas para eliminar
    if cols_to_drop:
        baseline_folds_metrics = baseline_folds_metrics.drop(columns=cols_to_drop)

flag_baseline_folds_metrics=False → se inicia entrenamiento por folds.


In [29]:
baseline_folds_metrics

""
CAT_baseline_fold_1
CAT_baseline_fold_2
CAT_baseline_fold_3
CAT_baseline_fold_4
CAT_baseline_fold_5


In [33]:
if flag_baseline_folds_metrics == False:
  save_metrics(baseline_folds_metrics, "/","0_baseline_folds_metrics")
else:
  print("Ya existen métricas del entrenamiento y están guardadas en disco")

Métricas guardadas en /content/drive/MyDrive/neural_profit/6_catboost_model///0_baseline_folds_metrics.parquet


In [34]:
def weighted_avg_metrics_from_df(df, w_train, w_valid, w_test):
    """
    Calcula el promedio ponderado de métricas a partir de un DataFrame
    con columnas del tipo train_RMSE, valid_RMSE, test_RMSE, etc.

    df : DataFrame con un fold por fila
    w_train, w_valid, w_test : pesos (cantidad de muestras por conjunto)
    """

    # Métricas base
    metricas = ["RMSE", "MAE", "R2", "SMAPE", "DirAcc"]

    resultados = {}

    for m in metricas:
        col_train = f"train_{m}"
        col_valid = f"valid_{m}"
        col_test  = f"test_{m}"

        # Promedio ponderado por fold, luego promedio entre folds
        valores_fold = (
            df[col_train] * w_train +
            df[col_valid] * w_valid +
            df[col_test]  * w_test
        ) / (w_train + w_valid + w_test)

        # Promedio total final entre folds
        resultados[m] = valores_fold.mean()

    return resultados

In [ ]:
if flag_baseline_metrics:
    print("flag_baseline_metrics=True → se omite el ponderado porque ya existe el dataset.")
else:
    print("flag_baseline_metrics=False → se inicia el ponderado")
    for k in k_folds:
        w = pesos_folds[k]

        # promedio ponderado para ESTE fold (sale como dict)
        res = weighted_avg_metrics_from_df(
            baseline_folds_metrics.loc[[f"baseline_fold_{k}"]],
            w["w_train"],
            w["w_valid"],
            w["w_test"]
        )

        # índice correspondiente en baseline_metrics
        idx = f"baseline_fold_{k}"

        # escribir directamente en el dataset transformers_metrics
        baseline_metrics.loc[idx, cols_base] = [res[m] for m in cols_base]



In [35]:
baseline_metrics

,RMSE,MAE,R2,SMAPE,DirAcc


In [36]:
if flag_baseline_metrics == False:
  save_metrics(baseline_metrics,"6_catboost_model","1_baseline_metrics")
else:
  print("Ya existen métricas del entrenamiento y están guardadas en disco")

OSError: Cannot save file into a non-existent directory: '/content/drive/MyDrive/neural_profit/6_catboost_model/6_catboost_model'

Este punto existe para garantizar reproducibilidad y continuidad del análisis sin reentrenar modelos cuando se pierden las métricas. Actúa como fallback: reconstruye la tabla de métricas de Random Forest a partir de valores ya validados y la persiste nuevamente, evitando el re-entrenamiento de los modelos.

Así, se mantiene la consistencia de resultados y la trazabilidad de comparaciones y conclusiones, incluso si el archivo original fue eliminado, corrompido o el entorno de ejecución cambió.